---
title : "Configuratie Agent"
execute:
    output: true
format:
    html:
        warning: false
        error: false
        code-fold: false
        css: style.css
        panel: center
---

De configuratie Agent van 'Talk with FEWS' heeft verschillende tools tot zijn beschikking.
Het taalmodel kan deze zelf invullen, maar hier wordt per tool een voorbeeld gegeven van de in en output. 

### Voorbereidingen
In eerste instantie worden de juiste variable opgehaald. Het context object bevat informatie over welke configuratie gebruikt wordt. In de voorbereiding is de configuratie al ingelezen in een database object. Dit is anders dan de normale 'FEWS database' waar alle data in zit. In deze notebook is alles met 'database' dus de configuratie database.


In [ ]:
from pydantic_ai import format_as_xml
from setup_quarto import setup_dot_env_docs

ctx = setup_dot_env_docs()

## Search workflows{#sec-search-workflows}
Om meer inzicht te krijgen in de configuratie van FEWS kan naar verschillende dingen gezocht worden.

De drie zoek mogelijkheden zijn:

- [`search_workflows`](#sec-search-workflows)
- [`search_timeseries`](#sec-search-timeseries)
- [`search_transformations`](#sec-search-transformations)

Workflows is een logisch start punt om de verschillende tools uit te leggen. 

Gegeven een (deel) van de workflow naam kan de informatie achter deze workflow worden opgehaald.

Bijvoorbeeld: een workflow die `im.wiwb.knmi` bevat, laten we zoek of er workflows zijn die deze naam bevatten.

In [ ]:
name_pattern = "knmi"

In [ ]:
from talk_with_fews.tools.config.workflow_search import search_workflows
from talk_with_fews.models import (
    WorkflowSearchOutput,
    WorkflowSearchInput,
)

workflow_search_input = WorkflowSearchInput(name_pattern=name_pattern)
workflow_search_output: WorkflowSearchOutput = await search_workflows(
    ctx=ctx, input=workflow_search_input
)

In [ ]:
print(format_as_xml(workflow_search_output, root_tag="WorkflowDetailsOutput"))

## Get workflow details
Gegeven een workflow ID (bijvoorbeeld uit `search_workflows`) kan de bijbehorende informatie over deze workflow worden opgehaald.

In [ ]:
workflow_id = workflow_search_output.results[0].id
workflow_id

In [ ]:
from talk_with_fews.tools.config.workflows_details import get_workflow_details
from talk_with_fews.models import (
    WorkflowDetailsOutput,
    WorkflowDetailsInput,
)

workflow_details_input = WorkflowDetailsInput(workflow_id=workflow_id)
workflow_details_output: WorkflowDetailsOutput = await get_workflow_details(
    ctx=ctx, input=workflow_details_input
)

In [ ]:
print(format_as_xml(workflow_details_output, root_tag="WorkflowDetailsOutput"))

## Get transformation details{#sec-get-transformation-details}
Gegeven de transformatie ID kunnen we kijken wat verder met deze workflow te maken heeft. Bijvoorbeeld uit tabel in de `get_workflow_details` tool. 

In [ ]:
transformation_id = 1357

In [ ]:
from talk_with_fews.tools.config.transformation_details import (
    get_transformation_details,
)
from talk_with_fews.models import (
    TransformationDetailsInput,
    TransformationDetailsOutput,
)

transformation_details_input = TransformationDetailsInput(
    transformation_id=transformation_id
)
transformation_details_output: TransformationDetailsOutput = (
    await get_transformation_details(ctx=ctx, input=transformation_details_input)
)

In [ ]:
print(
    format_as_xml(transformation_details_output, root_tag="TransformationDetailsOutput")
)

## Describe Timeseries
Gegeven een timeseries ID kan je bijbehorende informatie over de transformaties ophalen waar deze timeseries in voor komt. 
Hier gebruiken we een timeseries ID uit [`get_transformation_details`](#sec-get-transformation-details). 

In [ ]:
timeseries_id = transformation_details_output.results[0].time_series_links[0].tss_id
timeseries_id

In [ ]:
from talk_with_fews.tools.config.timeseries_describe import describe_timeseries
from talk_with_fews.models import (
    DescribeTimeSeriesInput,
    DescribeTimeSeriesOutput,
)

timeseries_describe_input = DescribeTimeSeriesInput(timeseries_id=timeseries_id)
timeseries_describe_output: DescribeTimeSeriesOutput = await describe_timeseries(
    ctx=ctx, input=timeseries_describe_input
)

In [ ]:
print(format_as_xml(timeseries_describe_output, root_tag="DescribeTimeSeriesOutput"))

## Search Timeseries{#sec-search-timeseries}
Naast zoeken op workflows en dan via transformaties naar de bijbehorende timeseries, zoals met [`search_workflow`](#sec-search-workflows), kan je ook direct zoeken op timeseries.

Dit kan op basis van:
- parameter_id
- qualifier_pattern
- module_instance
- endpoint
- include_transformations

Hier worden bijbehorende timeseries opgehaald die voldoen aan de opgegeven criteria.

In [ ]:
from talk_with_fews.tools.config.timeseries_search import search_timeseries
from talk_with_fews.models import (
    TimeSeriesSearchInput,
    TimeSeriesSearchOutput,
)

timeseries_search_input = TimeSeriesSearchInput(module_instance="im.wiwb.knmi")

timeseries_search_output: TimeSeriesSearchOutput = await search_timeseries(
    ctx=ctx, input=timeseries_search_input
)

In [ ]:
print(format_as_xml(timeseries_search_output, root_tag="TimeSeriesSearchOutput"))

## Search Transformation{#sec-search-transformations}
Als laatste van de searches: [`search_workflows`](#sec-search-workflows),[`search_workflows`](#sec-search-timeseries), kan je ook zoeken op transformaties. 
Dit is als configurator iets minder intuïtief, maar kan handig zijn voor de AI-agent om door te zoeken naar andere transformaties. 

Dit kan op basis van description_pattern, met als filter opties:
- workflow_name
- module_config
- max_depth

We gebruiken hier de zelfde name pattern als bij de workflow search, namelijk:

In [ ]:
name_pattern

In [ ]:
from talk_with_fews.tools.config.transformation_search import search_transformations
from talk_with_fews.models import (
    TransformationSearchInput,
    TransformationSearchOutput,
)

transformation_search_input = TransformationSearchInput(
    description_pattern=name_pattern
)
transformation_search_output: TransformationSearchOutput = await search_transformations(
    ctx=ctx, input=transformation_search_input
)

In [ ]:
print(format_as_xml(transformation_search_output, root_tag="TimeSeriesSearchOutput"))

## Get database schema
De tools hier boven waren vooral gericht op het ophalen van informatie over de configuratie.
Vanaf hier worden specifiekere tools besproken die helpen bij het werken met de configuratie database.
Om op de juiste manier de tool execute query te gebruiken, is het handig om te weten wat er in de database zit.
Hiervoor zijn de schema's van de configuratie database.
Deze tool geeft een overzicht van de beschikbare tabellen en kolommen.

In [ ]:
from talk_with_fews.tools.config.get_database_schema import get_database_schema
from talk_with_fews.models import (
    DatabaseSchemaOutput,
    DatabaseSchemaInput,
)

database_schema_input = DatabaseSchemaInput()

database_schema_output: DatabaseSchemaOutput = await get_database_schema(
    ctx=ctx, input=database_schema_input
)

In [ ]:
print(format_as_xml(database_schema_output, root_tag="DatabaseSchemaOutput"))

## Execute query
Deze tool geeft alle mogelijke vrijheid: het taalmodel kan elke SQL query uitvoeren.
Dit geeft het AI model alle vrijheid om de database te doorzoeken.

In [ ]:
from talk_with_fews.tools.config.sql_query import execute_query
from talk_with_fews.models import SQLQueryInput, SQLQueryOutput

sql_query_input = SQLQueryInput(
    query="SELECT * FROM trafos",
)

sql_query_output: SQLQueryOutput = await execute_query(ctx=ctx, input=sql_query_input)

In [ ]:
print(format_as_xml(sql_query_output, root_tag="SQLQueryOutput"))

## Find Workflow Call Hierarchy

Om de workflows te begrijpen is het handig om te weten welke workflows aan elkaar gerelateerd zijn.
Deze tool geeft een overzicht van de workflow hiërarchie, oftewel welke workflows aan elkaar gerelateerd zijn.
Dit kan lang door gaan, daarom is er een `max_depth` parameter die aangeeft hoe diep de hiërarchie moet worden opgehaald.

In [ ]:
root_workflow_id = workflow_search_output.results[0].id
root_workflow_id

In [ ]:
from talk_with_fews.tools.config.workflow_hierarchy import find_workflow_call_hierarchy
from talk_with_fews.models import (
    WorkflowCallHierarchyInput,
)
from talk_with_fews.models import SQLQueryOutput as sql_query_output

workflow_call_hierarchy_input = WorkflowCallHierarchyInput(
    root_workflow_id=root_workflow_id
)
workflow_call_hierarchy_output: sql_query_output = await find_workflow_call_hierarchy(
    ctx=ctx, input=workflow_call_hierarchy_input
)

In [ ]:
print(
    format_as_xml(
        workflow_call_hierarchy_output, root_tag="WorkflowCallHierarchyOutput"
    )
)

## Analyze Dependencies

Deze tool geeft een overzicht van de afhankelijkheden tussen de verschillende workflows.
Hierbij kun je aangeven welke richting van afhankelijkheden je wilt analyseren: upstream, downstream, of beide.
ook geef je aan welke entiteit je wilt analyseren: timeseries, transformation, of workflow.
Dit is eigenlijk een voorbereide versie van de `execute_query` tool, maar dan specifiek gericht op het analyseren van afhankelijkheden.

In [ ]:
from talk_with_fews.tools.config.dependency_analysis import analyze_dependencies
from talk_with_fews.models import (
    DependencyAnalysisInput,
    DependencyAnalysisOutput,
)

analyze_dependencies_input = DependencyAnalysisInput(
    entity_type="transformation",
    entity_id=1,
    direction="downstream",  # 'upstream', 'downstream', or 'both'
)

analyze_dependencies_output: DependencyAnalysisOutput = await analyze_dependencies(
    ctx=ctx, input=analyze_dependencies_input
)

In [ ]:
print(format_as_xml(analyze_dependencies_output, root_tag="DependencyAnalysisOutput"))

## Analyse Dead Points
Deze tool geeft een overzicht van de dode punten in de configuratie.
Dit is een uitgewerkte analyse tool die sp laat zien hoe we een speciefieke analyse kunnen uitvoeren op de configuratie database. 


In [ ]:
from talk_with_fews.tools.config.dead_point_analysis import analyze_dead_points
from talk_with_fews.models import (
    DeadPointsAnalysisInput,
    DeadPointsAnalysisOutput,
)

dead_point_analysis_input = DeadPointsAnalysisInput(
    entity_type="transformation",
    entity_id=1,
)
dead_point_analysis_output: DeadPointsAnalysisOutput = await analyze_dead_points(
    ctx=ctx, input=dead_point_analysis_input
)

In [ ]:
print(format_as_xml(dead_point_analysis_output, root_tag="DeadPointsAnalysisOutput"))

## Visualize graph{#sec-visualize-graph}

Hier wordt een grafische weergave van de workflow hiërarchie gegeven.

In [ ]:
from talk_with_fews.tools.config.graph_visualize import visualize_graph
from talk_with_fews.models import (
    GraphVisualizationInput,
    GraphVisualizationOutput,
)

graph_visualize_input = GraphVisualizationInput(
    root_entity_id=1, root_entity_type="workflow"
)

graph_visualize_output: GraphVisualizationOutput = await visualize_graph(
    ctx=ctx, input=graph_visualize_input
)

In [ ]:
print(format_as_xml(graph_visualize_output, root_tag="DeadPointsAnalysisOutput"))

De grafiek kunnen we vervolgens hier ook zien:

In [ ]:
from IPython.display import Markdown, display

display(Markdown(graph_visualize_output.markdown_content))

In [ ]:
from talk_with_fews.tools.config.parameter_search import search_parameters
from talk_with_fews.models import GetParameterInput, GetParameterOutput

get_parameter_input = GetParameterInput(number_of_parameters=10)

get_parameter_output: GetParameterOutput = await search_parameters(
    ctx, get_parameter_input
)

In [ ]:
print(format_as_xml(get_parameter_output, root_tag="GetParameterOutput"))